# 🌌 Stellar Class — 0.972 Weighted-Decorrelated Blend

**Playground Series S6E6** · metric = **balanced accuracy** · classes `GALAXY / QSO / STAR`.

This notebook reproduces a **top-region (~0.972)** submission by blending the strongest public
submissions with a **score-weighted, diversity-decorrelated vote**, then resolving the small
**disagreement region** with a probability consensus.

### How the leaderboard top is actually reached
Inspecting public submission histories (e.g. `zoli800`'s `submission_history.csv`) shows the very
top is reached by **coordinated leaderboard probing**: flip ~4 rows at a time
(`GALAXY↔STAR`, `GALAXY↔QSO`, `STAR↔GALAXY`), keep the change if the public score rises. The base
that probing starts from is a **blend of strong public models** — which is exactly what this
notebook builds.

### Add these datasets as inputs (Kaggle → Add Input)
- `nina2025/ps-s6e6` — pack of scored submissions (filenames = LB scores)
- `vladislavagamova/s6e6-097186-final-submission`
- `mehrankazeminia/s6e6-97181`
- `zoli800/s6e6-097209-final-submission` (optional, the current public best)
- `romonedunlop/s6e6-kaisei-error-features` (error-detector for the probing analysis)

> Full from-scratch pipeline (XGB/LGBM/Cat/HistGBM/MLP → TabPFN-3 meta) +
> AutoGluon variant: **https://github.com/MitudruDutta/Predicting-Stellar-Class**

In [1]:
import numpy as np, pandas as pd, glob, os
CLASSES = ['GALAXY','QSO','STAR']; C2I = {c:i for i,c in enumerate(CLASSES)}

# competition test ids (handle both Kaggle input layouts)
cand_comp = glob.glob('/kaggle/input/**/test.csv', recursive=True)
COMP_TEST = next((p for p in cand_comp if 'playground-series-s6e6' in p), cand_comp[0])
test = pd.read_csv(COMP_TEST); test_ids = test['id'].values; M = len(test_ids)
print('test:', COMP_TEST, M, 'rows')

def try_load_sub(path):
    """Return integer labels aligned to test_ids, or None if not a valid id,class submission."""
    try:
        d = pd.read_csv(path)
    except Exception:
        return None
    if 'id' not in d.columns or 'class' not in d.columns:
        return None
    if len(d) != M:
        return None
    d = d.sort_values('id').reset_index(drop=True)
    if not np.array_equal(d['id'].values, test_ids):
        return None
    if not d['class'].isin(CLASSES).all():
        return None
    return d['class'].map(C2I).values

# scan EVERY csv under /kaggle/input (recursive). Score = float(filename) when parseable,
# else a default 0.969 so unnamed strong subs still vote (lightly).
members = {}
for f in glob.glob('/kaggle/input/**/*.csv', recursive=True):
    if os.path.basename(f) in ('test.csv','train.csv','sample_submission.csv'): continue
    lab = try_load_sub(f)
    if lab is None: continue
    stem = os.path.splitext(os.path.basename(f))[0]
    try:
        sc = float(stem)
        if not (0.96 < sc < 0.98): sc = 0.969
    except ValueError:
        sc = 0.969
    name = f.split('/kaggle/input/')[-1]
    members[name] = (lab, sc)

# de-duplicate identical submissions, keep highest score; sort; cap at 15
uniq = {}
for n,(lab,sc) in members.items():
    key = lab.tobytes()
    if key not in uniq or sc > uniq[key][2]:
        uniq[key] = (n, lab, sc)
members = {n:(lab,sc) for n,lab,sc in uniq.values()}
members = dict(sorted(members.items(), key=lambda kv:-kv[1][1])[:15])

assert members, ('No submission CSVs found under /kaggle/input. Add datasets like '
                 'nina2025/ps-s6e6, zoli800/s6e6-097209-final-submission, etc. via Add Input.')
print(f'{len(members)} unique strong members:')
for n,(_,sc) in members.items(): print(f'  {sc:.5f}  {n}')

test: /kaggle/input/competitions/playground-series-s6e6/test.csv 247435 rows
15 unique strong members:
  0.97183  datasets/nina2025/ps-s6e6/0.97183.csv
  0.97182  datasets/nina2025/ps-s6e6/0.97182.csv
  0.97181  datasets/nina2025/ps-s6e6/0.97181.csv
  0.97179  datasets/nina2025/ps-s6e6/0.97179.csv
  0.97175  datasets/nina2025/ps-s6e6/0.97175.csv
  0.97171  datasets/nina2025/ps-s6e6/0.97171.csv
  0.97170  datasets/nina2025/ps-s6e6/0.97170.csv
  0.97150  datasets/nina2025/ps-s6e6/0.97150.csv
  0.97146  datasets/nina2025/ps-s6e6/0.97146.csv
  0.97144  datasets/nina2025/ps-s6e6/0.97144.csv
  0.97135  datasets/nina2025/ps-s6e6/0.97135.csv
  0.97133  datasets/nina2025/ps-s6e6/0.97133.csv
  0.97130  datasets/nina2025/ps-s6e6/0.97130.csv
  0.97129  datasets/nina2025/ps-s6e6/0.97129.csv
  0.97127  datasets/nina2025/ps-s6e6/0.97127.csv


## Weighted + decorrelated vote

Two ideas combined:
1. **Score weighting** — better submissions get more say (weight = score − min + ε).
2. **Decorrelation** — near-duplicate submissions are downweighted by their average agreement with
   the others, so a clique of identical files can't dominate the vote.

The blend can only change a row inside the **disagreement region** (where members differ), so we
also measure how small that region is.

In [2]:
names = list(members)
L = np.stack([members[n][0] for n in names])          # (k, M)
S = np.array([members[n][1] for n in names])
k = len(names)
unan = (L == L[0]).all(0)
print(f'members={k}  unanimous rows: {unan.mean()*100:.2f}%  | disagreement region: {(~unan).sum()} rows')

w = S - S.min() + 1e-4; w = w / w.sum()
if k >= 3:
    agr = np.array([[(L[a]==L[b]).mean() for b in range(k)] for a in range(k)])
    off = agr.copy(); np.fill_diagonal(off, np.nan)
    uniqf = 1.0 / np.nanmean(off, 1); uniqf /= uniqf.mean()
else:
    uniqf = np.ones(k)
wd = w * uniqf; wd /= wd.sum()

V = np.zeros((M, 3))
for j in range(k):
    np.add.at(V, (np.arange(M), L[j]), wd[j])
blend = V.argmax(1)
blend[unan] = L[0][unan]
print('blend class dist:', {CLASSES[i]: int((blend==i).sum()) for i in range(3)})
best_name = max(names, key=lambda n: members[n][1])
print(f'agreement with best single ({best_name}, {members[best_name][1]:.5f}):',
      round((blend==members[best_name][0]).mean(), 5))

members=15  unanimous rows: 99.79%  | disagreement region: 523 rows
blend class dist: {'GALAXY': 156958, 'QSO': 51451, 'STAR': 39026}
agreement with best single (datasets/nina2025/ps-s6e6/0.97183.csv, 0.97183): 0.99973


## Optional: probability-consensus tie-break on the disagreement region

Where members disagree, fall back to the **average class probability** if probability artifacts are
available (e.g. `cdeotte/s6e6-oof-and-test-preds`). Here we use the vote share itself as a soft
consensus — the larger the winning margin, the more confident the flip.

In [3]:
margin = V.max(1) - np.sort(V,1)[:,-2]
print('low-margin (contested) rows:', int((margin < 0.2).sum()))
final = blend.copy()
anchor = members[best_name][0]          # strongest single submission
unsure = (~unan) & (margin < 0.15)
final[unsure] = anchor[unsure]
print('final vs blend flips:', int((final!=blend).sum()))

low-margin (contested) rows: 64
final vs blend flips: 16


In [4]:
sub = pd.DataFrame({'id': test_ids, 'class': np.array(CLASSES)[final]})
sub.to_csv('submission.csv', index=False)
print('submission.csv written', sub.shape)
print(sub['class'].value_counts()); sub.head()

submission.csv written (247435, 2)
class
GALAXY    156958
QSO        51449
STAR       39028
Name: count, dtype: int64


,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


## Notes & honest caveats

- This blend lands in the **~0.972 public region**; the exact public #1 is reached by additional
  **leaderboard probing** (4-row flips kept if the LB rises) on top of a blend like this. Probing
  optimises the *public* split specifically — guard against private-LB shuffle by also keeping a
  robust single model for final selection.
- **`class_weight='balanced'`** is the key lever for balanced accuracy when training your own bases.
- Residual error is dominated by **low-redshift GALAXY/STAR** photometric degeneracy.

⭐ From-scratch pipeline + TabPFN-3 meta + AutoGluon + the full probing ledger:
**https://github.com/MitudruDutta/Predicting-Stellar-Class**